# Mini Decoded — Train `model2_mini_decoded` with History Encoder + Decoder

**Notebook Version**: 1.0  
**Purpose**: Train the `vector_joint_bc_transition_v1` family's history-decoded variant.  
**Research question**: *Does training with a history-reconstruction auxiliary improve the policy trunk on the flat-vector mini family, even when history is absent at inference?*

## Why this notebook exists

A prior local run (artifact `model2_mini_decoded_hist_20260426_045507`) reported `hist_decode_loss ≈ 0.00028` — the loss was meaningless. Root cause: `BattleStateTracker.apply_turn` cleared `_current_turn_events` *before* `iter_turn_examples` could yield it, so `build_sequence_vocab` saw zero unique event keys and persisted a 4-entry specials-only vocab. Every event in training mapped to `<UNK>` and the decoder learned the trivial `UNK→UNK` mapping.

The fix lives on branch `feature/claude-monitor` (BattleStateTracker no longer clears `_current_turn_events` at the tail of `apply_turn`). This notebook clones that branch, verifies the fix is present, runs training on Colab GPU (local CPU was ~10 min/epoch × 30 epochs = ~5 hours), and validates two diagnostic checkpoints:

1. **`sequence_vocab` size > 4** — confirms the tracker fix carried into vocab building.
2. **`hist_decode_loss` floor ≥ ~0.5 nats** — confirms the decoder is learning a non-trivial signal (uniform baseline over a ~6 K-class vocab is `ln(6325) ≈ 8.75`).

## Phase 0: GPU Verification & Cost Baseline

In [ ]:
# ── 0. GPU verify + cost baseline ─────────────────────────────────────────────
import subprocess, time

TRAINING_START_WALL = time.time()

gpu_info = {}
try:
    r = subprocess.run(
        ['nvidia-smi',
         '--query-gpu=name,memory.total,power.limit,driver_version',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5,
    )
    if r.returncode == 0:
        parts = [p.strip() for p in r.stdout.strip().split(',')]
        gpu_info = {
            'name':           parts[0] if len(parts) > 0 else 'unknown',
            'memory_total':   parts[1] if len(parts) > 1 else 'unknown',
            'power_limit_w':  parts[2] if len(parts) > 2 else 'unknown',
            'driver_version': parts[3] if len(parts) > 3 else 'unknown',
        }
except Exception as e:
    gpu_info = {'name': 'CPU', 'error': str(e)}

GPU_CU_PER_HOUR = {'T4': 1.96, 'A100': 12.25, 'V100': 6.86, 'L4': 4.90, 'CPU': 0.0}
gpu_name = gpu_info.get('name', 'CPU')
gpu_tier = next((k for k in GPU_CU_PER_HOUR if k in gpu_name.upper()), 'CPU')
CU_RATE  = GPU_CU_PER_HOUR[gpu_tier]

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')

print('=' * 70)
print('GPU BASELINE')
print('=' * 70)
for k, v in gpu_info.items():
    print(f'  {k:20s}: {v}')
print(f'  TF version          : {tf.__version__}')
print(f'  TF GPUs detected    : {[g.name for g in gpus]}')
print(f'  Cost tier           : {gpu_tier}  ({CU_RATE} CU/hr)')
print(f'  Wall-clock baseline : {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(TRAINING_START_WALL))}')
print('=' * 70)

if not gpus and gpu_tier == 'CPU':
    print()
    print('WARNING: No GPU — training will be very slow.')
    print('Colab: Runtime → Change runtime type → T4 GPU, then restart.')

In [ ]:
# ── 0b. Keep-alive ping (avoids Colab idle disconnect) ────────────────────────────
from IPython.display import Javascript, display

display(Javascript("""
  function keepColab() { document.querySelector('#top-toolbar')?.click(); }
  window._keepAlive = setInterval(keepColab, 55000);
  console.log('Keep-alive active. Stop: clearInterval(window._keepAlive)');
"""))
print('Keep-alive active (pings Colab every 55s).')

## Setup: Repo + Bug-Fix Verification

In [ ]:
# ── 1. Clone repo on the branch carrying the BattleStateTracker fix ─────────────────
import os, shutil

REPO_URL = 'https://github.com/AlterProgramming/Pokemon-Showdown-Agents-Go-Brrrr.git'
BRANCH   = 'feature/claude-monitor'   # carries core/BattleStateTracker.py fix
REPO_DIR = '/content/Pokemon-Showdown-Agents-Go-Brrrr'

os.chdir('/content')
if os.path.exists(REPO_DIR):
    print(f'Removing stale checkout: {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

!git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log --oneline -3

# Critical verification: the BattleStateTracker fix MUST be present.
# Bug signature: a `self._current_turn_events = []` reset AFTER the
# `_past_turn_events.append(...)` block in apply_turn (introduced by
# commit 4f00715, fixed in feature/claude-monitor).
tracker_src = open('core/BattleStateTracker.py').read()
fix_marker  = 'do not clear self._current_turn_events here'
if fix_marker not in tracker_src:
    raise RuntimeError(
        'BattleStateTracker fix marker not found. The clone is on a '
        'branch that still has the apply_turn bug — history training '
        'will silently produce a degenerate sequence_vocab. '
        f'Checked branch: {BRANCH}.'
    )
print('  ✓  BattleStateTracker apply_turn fix present')

# Regression test should also be present.
test_src = open('tests/test_battle_state_tracker.py').read()
if 'TurnEventsV1YieldTests' not in test_src:
    raise RuntimeError('Regression test TurnEventsV1YieldTests missing from this checkout.')
print('  ✓  TurnEventsV1YieldTests regression present')

In [ ]:
# ── 1b. Run the regression tests before spending GPU time ──────────────────────
!python3 -m pytest tests/test_battle_state_tracker.py tests/test_turn_event_v1.py tests/test_turn_event_tokenizer.py -q

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────────────────
import importlib, subprocess, tensorflow as tf
print('TF:', tf.__version__)

!pip install -q kagglehub

import keras as _keras_check
_ver = tuple(int(x) for x in _keras_check.__version__.split('.')[:2])
if _ver < (3, 14):
    print(f'Upgrading keras {_keras_check.__version__} → >=3.14.0 ...')
    subprocess.run(['pip', 'install', '-q', 'keras>=3.14.0'], check=True)
    importlib.invalidate_caches()

import keras
print(f'keras {keras.__version__} ✓')

In [ ]:
# ── 3. Download battle data ──────────────────────────────────────────────
import kagglehub, os, glob

DATASET = 'thephilliplin/pokemon-showdown-battles-gen9-randbats'
print('Downloading', DATASET, '...')
data_path = kagglehub.dataset_download(DATASET)

json_files = glob.glob(os.path.join(data_path, '*.json'))
print(f'Dataset path : {data_path}')
print(f'JSON files   : {len(json_files):,}')

## Training Configuration

Defaults reproduce the prior local run's hyperparameters exactly. Change `MAX_BATTLES` to a small value (e.g. 50) for a smoke test.

In [ ]:
# ── 4. Training configuration ─────────────────────────────────────────────
import os
from datetime import datetime

_ts = datetime.now().strftime('%Y%m%d_%H%M')
MODEL_VARIANT = 'model2_mini_decoded'
RUN_ID        = f'{MODEL_VARIANT}_{_ts}'
OUTPUT_DIR    = f'/content/training_artifacts/{RUN_ID}'

# ── Data ──────────────────────────────────────────────────────
MAX_BATTLES = 5000        # SMOKE TEST: set to 50 for ~3-min run
VAL_RATIO   = 0.2
SEED        = 42

# ── Optimiser ──────────────────────────────────────────────────
EPOCHS        = 30
BATCH_SIZE    = 256
LEARNING_RATE = 0.001

# ── Architecture ─────────────────────────────────────────────────
HIDDEN_DIM = 256
DEPTH      = 3
DROPOUT    = 0.1

# ── Auxiliary heads (mini family) ─────────────────────────────────────
PREDICT_TURN_OUTCOME  = True   # transition head
PREDICT_VALUE         = False  # not used in this variant
TRANSITION_WEIGHT     = 0.25

# ── History encoder + decoder ────────────────────────────────────────
PREDICT_FROM_HISTORY    = True
HISTORY_TURNS           = 8
HISTORY_EVENTS_PER_TURN = 24
HISTORY_EMBED_DIM       = 16
HISTORY_LSTM_DIM        = 32
HISTORY_DECODE_WEIGHT   = 0.1

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('=' * 70)
print(f'TRAINING CONFIG — {MODEL_VARIANT}')
print('=' * 70)
print(f'  Run ID        : {RUN_ID}')
print(f'  Output dir    : {OUTPUT_DIR}')
print(f'  Max battles   : {MAX_BATTLES}')
print(f'  Epochs        : {EPOCHS}')
print(f'  Batch / LR    : {BATCH_SIZE} / {LEARNING_RATE}')
print(f'  Architecture  : hidden={HIDDEN_DIM}  depth={DEPTH}  dropout={DROPOUT}')
print(f'  Transition    : {PREDICT_TURN_OUTCOME}  (w={TRANSITION_WEIGHT})')
print(f'  History       : K={HISTORY_TURNS}  E={HISTORY_EVENTS_PER_TURN}  '
      f'embed={HISTORY_EMBED_DIM}  lstm={HISTORY_LSTM_DIM}  decode_w={HISTORY_DECODE_WEIGHT}')
print('=' * 70)

## Auto-Save: Periodic GCS Sync

Spawns a background daemon thread that uploads `OUTPUT_DIR` to `gs://artifacts-model-serving/artifacts/model2_mini_decoded/<run_id>/` every 5 minutes during training. Survives Colab tab disconnects, runtime preemption, and OOM kills — anything written to disk gets shipped within ~5 min.

Auth: requires Colab secret `GCS_SA_KEY_JSON` (Colab → 🔑 sidebar). If absent, falls back to interactive `auth.authenticate_user()` so the cell still runs.

In [ ]:
# ── 4b. Init GCS auth + start periodic sync daemon ─────────────────────────────
import json, os, threading
from google.cloud import storage

BUCKET     = 'artifacts-model-serving'
GCS_PREFIX = f'artifacts/model2_mini_decoded/{RUN_ID}'
GCS_SYNC_INTERVAL_SEC = 300   # every 5 min

_gcs_credentials = None
try:
    from google.colab import userdata
    _sa_json = userdata.get('GCS_SA_KEY_JSON')
    from google.oauth2 import service_account
    _gcs_credentials = service_account.Credentials.from_service_account_info(
        json.loads(_sa_json),
        scopes=['https://www.googleapis.com/auth/cloud-platform'],
    )
    print('Using service account credentials from Colab secret GCS_SA_KEY_JSON.')
except Exception as _sa_err:
    print(f'Secret unavailable ({type(_sa_err).__name__}) — falling back to authenticate_user().')
    from google.colab import auth
    auth.authenticate_user()

_gcs_client = storage.Client(project='pokemon-battle-agent-494015', credentials=_gcs_credentials)
_gcs_bucket = _gcs_client.bucket(BUCKET)

_uploaded_mtimes = {}   # local_path → mtime at last upload
_sync_lock       = threading.Lock()

def gcs_sync_dir(verbose=True):
    """Upload any file in OUTPUT_DIR newer than its last upload. Idempotent."""
    if not os.path.isdir(OUTPUT_DIR):
        return 0, 0
    uploaded = 0
    skipped  = 0
    with _sync_lock:
        for fname in sorted(os.listdir(OUTPUT_DIR)):
            local_path = os.path.join(OUTPUT_DIR, fname)
            if not os.path.isfile(local_path):
                continue
            mtime = os.path.getmtime(local_path)
            if _uploaded_mtimes.get(local_path) == mtime:
                skipped += 1
                continue
            try:
                blob_name = f'{GCS_PREFIX}/{fname}'
                _gcs_bucket.blob(blob_name).upload_from_filename(local_path)
                _uploaded_mtimes[local_path] = mtime
                uploaded += 1
                if verbose:
                    size_kb = os.path.getsize(local_path) / 1024
                    print(f'  [gcs] ↑ {fname}  ({size_kb:.0f} KB)')
            except Exception as e:
                print(f'  [gcs] FAILED {fname}: {e}')
    return uploaded, skipped

_sync_stop = threading.Event()

def _sync_loop():
    while not _sync_stop.wait(GCS_SYNC_INTERVAL_SEC):
        try:
            up, sk = gcs_sync_dir(verbose=False)
            if up:
                print(f'  [gcs sync] {up} new/changed file(s) uploaded, {sk} unchanged')
        except Exception as e:
            print(f'  [gcs sync] error: {e}')

_sync_thread = threading.Thread(target=_sync_loop, daemon=True, name='gcs_sync')
_sync_thread.start()
print(f'GCS sync daemon started — uploads to gs://{BUCKET}/{GCS_PREFIX}/ every {GCS_SYNC_INTERVAL_SEC}s.')
print('Survives Colab disconnects: anything on disk gets shipped within 5 min.')

## Phase 1: Train

In [ ]:
# ── 5. Train ───────────────────────────────────────────────────────────
import subprocess, sys, os, re, time

env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + os.pathsep + env.get('PYTHONPATH', '')

cmd = [
    sys.executable, '-u', 'train_policy_mini.py',
    '--model-variant',           MODEL_VARIANT,
    '--output-dir',              OUTPUT_DIR,
    '--max-battles',             str(MAX_BATTLES),
    '--val-ratio',               str(VAL_RATIO),
    '--seed',                    str(SEED),
    '--epochs',                  str(EPOCHS),
    '--batch-size',              str(BATCH_SIZE),
    '--learning-rate',           str(LEARNING_RATE),
    '--hidden-dim',              str(HIDDEN_DIM),
    '--depth',                   str(DEPTH),
    '--dropout',                 str(DROPOUT),
    '--include-switches',
    '--transition-weight',       str(TRANSITION_WEIGHT),
]

if PREDICT_TURN_OUTCOME:
    cmd.append('--predict-turn-outcome')
if PREDICT_VALUE:
    cmd.append('--predict-value')
if PREDICT_FROM_HISTORY:
    cmd += [
        '--predict-from-history',
        '--history-turns',           str(HISTORY_TURNS),
        '--history-events-per-turn', str(HISTORY_EVENTS_PER_TURN),
        '--history-embed-dim',       str(HISTORY_EMBED_DIM),
        '--history-lstm-dim',        str(HISTORY_LSTM_DIM),
        '--history-decode-weight',   str(HISTORY_DECODE_WEIGHT),
    ]

EPOCH_HDR = re.compile(r'Epoch\s+\d+/\d+')
STEP_RE   = re.compile(r'^\s*(\d+)/(\d+)')
METRIC_RE = re.compile(r'loss|accuracy|mae|brier', re.IGNORECASE)
ERROR_RE  = re.compile(r'error|traceback|exception|failed|killed', re.IGNORECASE)

epoch_times  = []
_epoch_start = None

train_wall_start = time.time()
print('Starting training...')
print('Command:', ' '.join(cmd[:4]), '... [+', len(cmd) - 4, 'args]')
print('=' * 70)

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=REPO_DIR, env=env,
)

epoch_total_steps = None
mid_epoch_printed = False
suppressed_lines  = []

try:
    for raw_line in proc.stdout:
        line = raw_line.rstrip()
        if not line:
            continue
        if ERROR_RE.search(line):
            print(line); suppressed_lines = []; continue
        if EPOCH_HDR.match(line):
            if _epoch_start is not None:
                epoch_times.append(time.time() - _epoch_start)
            _epoch_start = time.time()
            print(); print(line)
            epoch_total_steps = None; mid_epoch_printed = False
            continue
        step_m = STEP_RE.match(line)
        if step_m and METRIC_RE.search(line):
            current_step = int(step_m.group(1))
            total_steps  = int(step_m.group(2))
            suppressed_lines.append(line)
            if epoch_total_steps != total_steps:
                epoch_total_steps = total_steps
            is_final = (current_step == total_steps)
            is_mid   = not mid_epoch_printed and current_step >= total_steps // 2 and not is_final
            if is_mid:
                pct = int(100 * current_step / total_steps)
                print(f'  [{pct:3d}%] {line.strip()}')
                mid_epoch_printed = True
            elif is_final:
                print(f'  {line.strip()}')
            continue
        suppressed_lines.append(line)

    if _epoch_start is not None:
        epoch_times.append(time.time() - _epoch_start)

    proc.wait()
finally:
    # Always stop the sync daemon and flush whatever's on disk one last time,
    # even if we exit via exception or KeyboardInterrupt.
    _sync_stop.set()
    print('\n[gcs] Final flush...')
    up, sk = gcs_sync_dir(verbose=True)
    print(f'[gcs] Final flush: {up} uploaded, {sk} unchanged.')

TRAINING_ELAPSED_SEC = time.time() - train_wall_start

print()
print('=' * 70)
print(f'Exit code        : {proc.returncode}')
print(f'Total wall time  : {TRAINING_ELAPSED_SEC/60:.1f} min  ({TRAINING_ELAPSED_SEC:.0f}s)')
if epoch_times:
    print(f'Epochs completed : {len(epoch_times)}')
    print(f'Avg epoch time   : {sum(epoch_times)/len(epoch_times):.1f}s')
print('=' * 70)

if proc.returncode != 0:
    print('TRAINING FAILED — last lines:')
    for l in suppressed_lines[-60:]:
        print(l)
    raise RuntimeError(f'Training exited with code {proc.returncode}')

## Phase 2: Cost Report

In [ ]:
# ── 6. Cost report ─────────────────────────────────────────────────────────
import json, os, subprocess

elapsed_hr   = TRAINING_ELAPSED_SEC / 3600
compute_units = elapsed_hr * CU_RATE

gpu_util = {}
try:
    r = subprocess.run(
        ['nvidia-smi',
         '--query-gpu=utilization.gpu,utilization.memory,memory.used,memory.total,power.draw',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5,
    )
    if r.returncode == 0:
        parts = [p.strip() for p in r.stdout.strip().split(',')]
        gpu_util = {
            'gpu_utilisation': parts[0] if len(parts) > 0 else '?',
            'mem_utilisation': parts[1] if len(parts) > 1 else '?',
            'mem_used':        parts[2] if len(parts) > 2 else '?',
            'mem_total':       parts[3] if len(parts) > 3 else '?',
            'power_draw_w':    parts[4] if len(parts) > 4 else '?',
        }
except Exception:
    pass

cost_report = {
    'gpu_name':                gpu_info.get('name', 'unknown'),
    'gpu_tier':                gpu_tier,
    'cu_rate_per_hr':          CU_RATE,
    'training_elapsed_sec':    round(TRAINING_ELAPSED_SEC, 1),
    'training_elapsed_min':    round(TRAINING_ELAPSED_SEC / 60, 2),
    'estimated_compute_units': round(compute_units, 3),
    'epochs_completed':        len(epoch_times),
    'epoch_times_sec':         [round(t, 1) for t in epoch_times],
    'avg_epoch_sec':           round(sum(epoch_times) / len(epoch_times), 1) if epoch_times else None,
    'gpu_snapshot':            gpu_util,
}
cost_path = os.path.join(OUTPUT_DIR, f'training_cost_{RUN_ID}.json')
json.dump(cost_report, open(cost_path, 'w'), indent=2)

print('=' * 70)
print('TRAINING COST REPORT')
print('=' * 70)
print(f'  GPU                    : {gpu_info.get("name", "unknown")}  ({gpu_tier})')
print(f'  Wall time              : {TRAINING_ELAPSED_SEC/60:.1f} min')
print(f'  Compute units (Colab)  : {compute_units:.3f}  CU')
print(f'  Epochs completed       : {len(epoch_times)}')
if epoch_times:
    print(f'  Avg epoch              : {sum(epoch_times)/len(epoch_times):.1f}s')
if gpu_util:
    print()
    print('  Post-training GPU snapshot:')
    for k, v in gpu_util.items():
        print(f'    {k:20s}: {v}')
print(f'\n  Saved → {cost_path}')
print('=' * 70)

## Phase 3: Bug-Fix Validation — the diagnostics that matter

These two cells are the whole point of the retrain. If either fails, the artifact is degenerate (same as the local 0.00028-loss run) and the fix didn't take.

In [ ]:
# ── 7. Validate sequence_vocab is non-trivial ────────────────────────────────
import os, json

artifacts = sorted(os.listdir(OUTPUT_DIR))
vocab_files = [f for f in artifacts if 'sequence_vocab' in f and f.endswith('.json')]
if not vocab_files:
    raise RuntimeError(
        'No sequence_vocab*.json in OUTPUT_DIR. The trainer did not write one — '
        'either --predict-from-history was off or the saver path changed.'
    )
vocab_path = os.path.join(OUTPUT_DIR, vocab_files[0])
vocab      = json.load(open(vocab_path))
size       = len(vocab)
size_bytes = os.path.getsize(vocab_path)

print(f'sequence_vocab path : {vocab_path}')
print(f'sequence_vocab size : {size}  ({size_bytes:,} bytes on disk)')
print(f'first 4 keys        : {list(vocab.keys())[:4]}   <- specials')
print(f'next 5 keys         : {list(vocab.keys())[4:9]}  <- real event keys')

if size <= 4:
    raise RuntimeError(
        f'sequence_vocab has only {size} entries (specials only). '
        'The BattleStateTracker fix did not take — history training will be '
        'degenerate. Investigate before benchmarking.'
    )
print()
print(f'  ✓  sequence_vocab is non-trivial ({size} entries; expected ~6000+)')

In [ ]:
# ── 8. Validate hist_decode_loss curve is non-degenerate ──────────────────────
import os, json, math

summary_files = [f for f in artifacts if f.startswith('evaluation_summary') and f.endswith('.json')]
manifest_files = [f for f in artifacts if f.startswith('run_manifest') and f.endswith('.json')]

best = None
first = None
last = None
n_epochs = None

if manifest_files:
    manifest = json.load(open(os.path.join(OUTPUT_DIR, manifest_files[0])))
    snap = manifest.get('evaluation_snapshot', {})
    metrics = snap.get('metrics', {})
    hd = metrics.get('hist_decode_loss')
    if hd:
        first    = hd.get('first')
        last     = hd.get('last')
        best     = hd.get('best')
        n_epochs = hd.get('count')

uniform_baseline = math.log(size)  # ln(vocab_size); random guess
trivial_floor    = 0.01            # below this, suspect UNK→UNK collapse

print('=' * 70)
print('hist_decode_loss CURVE')
print('=' * 70)
print(f'  Uniform baseline (ln {size})  : {uniform_baseline:.3f} nats')
print(f'  Epochs in summary             : {n_epochs}')
print(f'  First-epoch loss              : {first}')
print(f'  Last-epoch loss               : {last}')
print(f'  Best loss (any epoch)         : {best}')
print()

if best is None:
    print('WARNING: could not find hist_decode_loss in run_manifest. '
          'Check evaluation_summary or training logs manually.')
elif best < trivial_floor:
    raise RuntimeError(
        f'hist_decode_loss collapsed to {best:.6f} (< {trivial_floor}). '
        f'Likely the same UNK→UNK degeneracy as the prior run, despite '
        f'a {size}-entry vocab. Investigate the encoding path — the '
        f'decoder may not be reading event_history_tokens correctly.'
    )
else:
    print(f'  ✓  hist_decode_loss best={best:.4f} — non-degenerate, real signal')
    print(f'      (started near ln({size})={uniform_baseline:.2f}, '
          f'converged to {last:.4f})')

## Phase 4: Artifacts Summary

In [ ]:
# ── 9. List artifacts + show metadata ───────────────────────────────────────
import os, json

print('=' * 70)
print('ARTIFACTS')
print('=' * 70)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    size_str = f'{size/(1024*1024):.2f} MB' if size > 1024 * 1024 else f'{size/1024:.1f} KB'
    print(f'  {f:60s}  {size_str:>10}')

meta_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('training_metadata')]
if meta_files:
    meta = json.load(open(os.path.join(OUTPUT_DIR, meta_files[0])))
    print()
    print('TRAINING METADATA (excerpt)')
    print('-' * 70)
    keys = [
        'model_name', 'model_release_id', 'family_name', 'training_regime',
        'objective_set', 'epochs_completed', 'epochs_requested',
        'train_examples', 'val_examples',
        'use_history', 'history_turns', 'history_events_per_turn',
        'history_embed_dim', 'history_lstm_dim', 'history_decode_weight',
    ]
    for k in keys:
        if k in meta:
            print(f'  {k:30s}: {meta[k]}')

## Phase 5: Push Artifacts to GCS

Uploads the run dir to `gs://artifacts-model-serving/artifacts/model2_mini_decoded/<run_id>/`.

Set Colab secret `GCS_SA_KEY_JSON` once (Colab → 🔑 sidebar) with the service account JSON; otherwise falls back to `auth.authenticate_user()`.

In [ ]:
# ── 10. Final GCS push (backstop for cost report + post-train artifacts) ─────
# The sync daemon already shipped most files mid-training. This cell catches
# anything written AFTER training ended (cost report, validation outputs).
try:
    up, sk = gcs_sync_dir(verbose=True)
    print(f'\nFinal push: {up} uploaded, {sk} already in sync.')
    print(f'All artifacts at: gs://{BUCKET}/{GCS_PREFIX}/')
    print('\nNext steps locally:')
    print('  1. /gcs-push  to sync model_registry.json')
    print('  2. Inspect sequence_vocab size and hist_decode_loss curve')
    print('  3. If loss is real, plan Cell B/C benchmarking')
except Exception as e:
    print(f'GCS push failed: {e}')
    print('Artifacts remain at:', OUTPUT_DIR)